# Can we predict `passed_quiz`?
Protocol: `protocol.yaml`. Holdout rows from `holdout.csv`, 5-fold GroupKFold on `student_key` inside training rows, model chosen by mean CV ROC AUC, holdout scored once.

**Feature timing (columns.csv).** `student_key` is excluded (entity key). `session_minutes` is the length of the whole session, which is closed by the quiz, so it is only known once the quiz is over: excluded as not available at prediction time (it is also the strongest correlate, a sign of leakage). `hints_used`, `prior_score`, `enrol_year`, `school_code` are known before the quiz result.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
train_mask=~df.index.isin(hold_idx)
train=df[train_mask]; hold=df.loc[hold_idx]
assert len(hold)==2465 and set(train.student_key).isdisjoint(hold.student_key)
print(len(train),len(hold))

In [ ]:
TARGET='passed_quiz'
NUM=['prior_score','hints_used','enrol_year']
CAT=['school_code']
FEATURES=NUM+CAT
X_tr,y_tr,g_tr=train[FEATURES],train[TARGET],train['student_key']
X_ho,y_ho=hold[FEATURES],hold[TARGET]
def prep(scale=True):
    return ColumnTransformer([('num',StandardScaler() if scale else 'passthrough',NUM),
                              ('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=20,sparse_output=False),CAT)])
# all candidates named before any is scored; preprocessing is inside each pipeline (fit per fold)
candidates={
 'baseline_prior':Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=2000))]),
 'logreg_C1':Pipeline([('p',prep()),('m',LogisticRegression(C=1.0,max_iter=2000))]),
 'hgb_d3':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,random_state=0))]),
 'hgb_d5':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=5,learning_rate=0.05,max_iter=150,random_state=0))]),
}
print(list(candidates))

In [ ]:
cv=GroupKFold(n_splits=5)
folds=list(cv.split(X_tr,y_tr,g_tr))
cv_scores={}
for name,est in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(X_tr.iloc[tr],y_tr.iloc[tr])
        s.append(float(roc_auc_score(y_tr.iloc[va],m.predict_proba(X_tr.iloc[va])[:,1])))
    cv_scores[name]=s
cv_table=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cv_table

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
print('chosen:',chosen)
final=clone(candidates[chosen]).fit(X_tr,y_tr)   # refit on all training rows
ho_auc=float(roc_auc_score(y_ho,final.predict_proba(X_ho)[:,1]))  # scored once
print('holdout roc_auc',ho_auc)

In [ ]:
pos=lambda idx:[int(i) for i in idx]
evidence={
 'features':FEATURES,
 'candidates':list(candidates),
 'cv_folds':[[pos(train.index[tr]),pos(train.index[va])] for tr,va in folds],
 'cv_scores':cv_scores,
 'chosen':chosen,
 'holdout_rows':pos(hold_idx),
 'holdout_scores':{chosen:ho_auc},
}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the CV table and the holdout ROC AUC above: the headline is the holdout AUC of the CV-chosen model. With only pre-quiz features the signal is weak; interpret AUC near 0.5 as little predictability.